# Notebook 43 — Balanced-SupCon weight and routing-reliability finishing study

This is a validation-first, paired-seed follow-up to Notebook 42. Phase 1 changes only the class-reliability penalty from `1e-4` to `1e-3` while reusing the exact Notebook-40 Balanced-SupCon Stage-A/B artifacts. Phase 2 selects that penalty using paired validation macro-F1, fixes it, and brackets the current Stage-A Balanced-SupCon coefficient `0.10` with `0.05` and `0.20`. Test metrics are descriptive and never participate in either selection.

The deployed topology remains `47 → 128 → 64`, four linear experts, a frozen encoder/expert body in Stage C, a `64 → 32 → 4` gate, and the zero-initialized `4 × 22` class-reliability matrix.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS = 2  # Initial paired decision set; reserve seed 2 for confirmation.
RUN_TESTS = True
# ==================================================================


## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Fixed study definitions and contract preflight

All five candidate configurations are explicit repository files. Only the `0.10 / 1e-3` reliability candidate reuses Stage A/B. The four weight candidates intentionally retrain Stage A and B because changing the representation coefficient changes the encoder and therefore the experts. Only the two candidates matching the validation-selected reliability value are executed.

In [ ]:
import copy, torch, yaml
from models.encoder import build_encoder
from training.compact_soft_moe_run import CompactSoftMoEStudy
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')

BASELINE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_class_conditional_deeper_gate32_frozen_stagec_v1'
BASELINE_CONFIG = 'config/compact_soft_moe_balanced_supcon_class_conditional_frozen_deeper_gate_3seed.yaml'
RELIABILITY_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_class_conditional_deeper_gate32_frozen_stagec_rel1e3_v1'
RELIABILITY_CONFIG = 'config/compact_soft_moe_balanced_supcon_class_conditional_rel1e3_frozen_deeper_gate_3seed.yaml'
WEIGHT_RUNS = {
    0.0001: {
        0.05: ('nfv3_4way_moe_soft_balanced_supcon_w0p05_class_conditional_deeper_gate32_frozen_stagec_rel1e4_v1', 'config/compact_soft_moe_balanced_supcon_class_conditional_w0p05_rel1e4_3seed.yaml'),
        0.20: ('nfv3_4way_moe_soft_balanced_supcon_w0p20_class_conditional_deeper_gate32_frozen_stagec_rel1e4_v1', 'config/compact_soft_moe_balanced_supcon_class_conditional_w0p20_rel1e4_3seed.yaml'),
    },
    0.001: {
        0.05: ('nfv3_4way_moe_soft_balanced_supcon_w0p05_class_conditional_deeper_gate32_frozen_stagec_rel1e3_v1', 'config/compact_soft_moe_balanced_supcon_class_conditional_w0p05_rel1e3_3seed.yaml'),
        0.20: ('nfv3_4way_moe_soft_balanced_supcon_w0p20_class_conditional_deeper_gate32_frozen_stagec_rel1e3_v1', 'config/compact_soft_moe_balanced_supcon_class_conditional_w0p20_rel1e3_3seed.yaml'),
    },
}
all_runs = [(RELIABILITY_PREFIX, RELIABILITY_CONFIG)] + [item for runs in WEIGHT_RUNS.values() for item in runs.values()]
baseline = yaml.safe_load(Path(BASELINE_CONFIG).read_text())
for prefix, config_path in all_runs:
    study = yaml.safe_load(Path(config_path).read_text()); backbone = study['backbone']; training = backbone['training']
    assert study['seeds'] == [0, 1, 2] and study['expected_total_parameters'] == 22420
    assert backbone['architecture'] == 'moe_dataset_class_conditional'
    assert backbone['model']['encoder']['hidden_dims'] == [128] and backbone['model']['expert']['hidden_dims'] == []
    assert backbone['model']['gate']['hidden_dims'] == [32]
    assert training['representation']['objective'] == 'balanced_supcon' and training['representation']['temperature'] == 0.1
    assert training['stage_c_unfreeze'] == 'none' and training['stage_c']['freeze_experts'] is True
    assert training['stage_c']['optimizer']['lr'] == 0.001
    model_cfg = backbone['model']; model = build_model(backbone['architecture'], build_encoder(47, 64, model_cfg['encoder']), ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], model_cfg)
    apply_stage_c_trainability(model, backbone)
    assert model.class_reliability.shape == (4, 22) and torch.count_nonzero(model.class_reliability).item() == 0
    assert sum(p.numel() for p in model.parameters()) == 22420 and sum(p.numel() for p in model.parameters() if p.requires_grad) == 2300
rel = yaml.safe_load(Path(RELIABILITY_CONFIG).read_text())
assert rel['reuse_stage_a_from_prefix'] == rel['reuse_stage_b_from_prefix'] == 'nfv3_4way_moe_soft_balanced_supcon_deeper_gate32_frozen_stagec_v1'
base_body = copy.deepcopy(baseline['backbone']); rel_body = copy.deepcopy(rel['backbone'])
assert base_body['training']['stage_c'].pop('lambda_reliability') == 0.0001
assert rel_body['training']['stage_c'].pop('lambda_reliability') == 0.001
assert base_body == rel_body, 'Reliability phase may change only lambda_reliability'
for reliability, runs in WEIGHT_RUNS.items():
    for weight, (_prefix, config_path) in runs.items():
        study = yaml.safe_load(Path(config_path).read_text())
        assert 'reuse_stage_a_from_prefix' not in study and 'reuse_stage_b_from_prefix' not in study
        assert study['backbone']['training']['representation']['weight'] == weight
        assert study['backbone']['training']['stage_c']['lambda_reliability'] == reliability
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
print('Preflight passed: weight variants retrain A/B/C; reliability-only variant schedules C.')


## Dry-run, validation-only reliability selection, and execution

Every configuration is dry-run first. With `EXECUTE=True`, the `1e-3` reliability candidate runs first. The notebook then uses only the common paired seeds' validation macro-F1 to choose between `1e-4` and `1e-3`, and executes the `0.05` and `0.20` representation weights at that fixed value. A tie retains `1e-4`, the simpler incumbent.

In [ ]:
def study_command(config_path, prefix):
    return [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', config_path, '--prefix', prefix]

def run_logged(config_path, prefix, label, execute=False):
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    command = study_command(config_path, prefix)
    suffix = 'Training_Stream' if execute else 'Dry_Run'
    if execute: command += ['--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)]
    run_streaming_logged(command, str(summary_dir / f'Notebook_{suffix}.log'), env=os.environ.copy(), heartbeat_seconds=30, label=label)

for prefix, config_path in all_runs:
    runner_check = CompactSoftMoEStudy(base_config_path='config/default.yaml', study_config_path=config_path, prefix=prefix, execute=False)
    expected_stages = ['C'] if config_path == RELIABILITY_CONFIG else ['A', 'B', 'C']
    assert runner_check._final_nested(0)['training']['stages'] == expected_stages
    run_logged(config_path, prefix, f'43-dry-{Path(config_path).stem}', execute=False)

def completed_validation_macro(prefix):
    values = {}
    for seed in (0, 1, 2):
        path = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}' / 'Validation_Overall_Metrics.csv'
        manifest = path.parent / 'manifest.json'
        if path.is_file() and manifest.is_file():
            import pandas as pd
            values[seed] = float(pd.read_csv(path).iloc[0]['macro_f1'])
    return values

if EXECUTE:
    run_logged(RELIABILITY_CONFIG, RELIABILITY_PREFIX, '43-reliability-1e3', execute=True)
baseline_scores = completed_validation_macro(BASELINE_PREFIX)
candidate_scores = completed_validation_macro(RELIABILITY_PREFIX)
reliability_common = sorted(set(baseline_scores) & set(candidate_scores))
selected_reliability = None
if len(reliability_common) >= 2:
    baseline_mean = sum(baseline_scores[s] for s in reliability_common) / len(reliability_common)
    candidate_mean = sum(candidate_scores[s] for s in reliability_common) / len(reliability_common)
    selected_reliability = 0.001 if candidate_mean > baseline_mean else 0.0001
    print('Reliability paired seeds:', reliability_common, 'means:', {0.0001: baseline_mean, 0.001: candidate_mean}, 'selected:', selected_reliability)
else:
    print('Reliability selection unavailable: two common completed validation seeds are required.')
if EXECUTE and selected_reliability is None:
    raise RuntimeError('Phase 1 did not produce two paired validation results; weight sweep was not started.')
if EXECUTE:
    for weight, (prefix, config_path) in WEIGHT_RUNS[selected_reliability].items():
        run_logged(config_path, prefix, f'43-bsupcon-w{weight:g}-rel{selected_reliability:g}', execute=True)
else:
    print('NO TRAINING WAS STARTED. Review every dry-run, set EXECUTE=True, and rerun this cell.')


## Paired validation report and descriptive locked-test report

The helper below sets provenance columns rather than inserting them. This is intentionally compatible with both older CSVs that lack `seed` and newer `Resource_Accounting.csv` files that already contain it.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display

def with_provenance(frame, condition, seed):
    result = frame.copy(); result['condition'] = condition; result['seed'] = int(seed)
    leading = ['condition', 'seed']
    return result[leading + [column for column in result.columns if column not in leading]]

def load_condition(condition, prefix):
    validation_rows = []; test_rows = []; class_rows = []; dataset_rows = []; resource_rows = []; latent_rows = []
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Class_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Resource_Accounting.csv']
        if not all((result_dir / name).is_file() for name in required): continue
        overall = pd.read_csv(result_dir / 'Validation_Overall_Metrics.csv').iloc[0]
        per_dataset = pd.read_csv(result_dir / 'Validation_Per_Dataset_Metrics.csv')
        per_class = pd.read_csv(result_dir / 'Validation_Per_Class_Metrics.csv'); present = per_class['support'] > 0
        rare = per_class[(per_class['support'] >= 20) & (per_class['support'] < 200)]
        validation_rows.append({'condition': condition, 'seed': seed, 'validation_macro_f1': overall['macro_f1'], 'validation_accuracy': overall['accuracy'], 'validation_weighted_f1': np.average(per_class.loc[present, 'f1'], weights=per_class.loc[present, 'support']), 'validation_macro_precision': overall['macro_precision'], 'validation_macro_recall': overall['macro_recall'], 'worst_dataset_macro_f1': per_dataset['macro_f1'].min(), 'rare_recall_mean': rare['recall'].mean() if len(rare) else np.nan, 'weakest_class_recall': per_class.loc[present, 'recall'].min()})
        test_rows.append(with_provenance(pd.read_csv(result_dir / 'Overall_Metrics.csv').query("origin == 'ALL'"), condition, seed))
        class_rows.append(with_provenance(pd.read_csv(result_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL'"), condition, seed))
        dataset_rows.append(with_provenance(pd.read_csv(result_dir / 'Per_Dataset_Metrics.csv'), condition, seed))
        resource_rows.append(with_provenance(pd.read_csv(result_dir / 'Resource_Accounting.csv'), condition, seed))
        latent_path = result_dir / 'latent' / 'stage_A' / 'Latent_Probe_Metrics.csv'
        if latent_path.is_file(): latent_rows.append(with_provenance(pd.read_csv(latent_path), condition, seed))
    return validation_rows, test_rows, class_rows, dataset_rows, resource_rows, latent_rows

reliability_conditions = {'rel_1e4': BASELINE_PREFIX, 'rel_1e3': RELIABILITY_PREFIX}
loaded = {name: load_condition(name, prefix) for name, prefix in reliability_conditions.items()}
reliability_validation = pd.DataFrame([row for value in loaded.values() for row in value[0]])
if reliability_validation.empty: print('No completed reliability cells exist yet.')
else:
    common = sorted(set(reliability_validation.query("condition == 'rel_1e4'")['seed']) & set(reliability_validation.query("condition == 'rel_1e3'")['seed']))
    print('Reliability common paired seeds:', common)
    reliability_paired = reliability_validation.query('seed in @common').copy(); display(reliability_paired)
    for metric in ['validation_macro_f1', 'validation_accuracy', 'validation_weighted_f1', 'validation_macro_precision', 'validation_macro_recall', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']:
        pivot = reliability_paired.pivot(index='seed', columns='condition', values=metric); pivot['delta_1e3_minus_1e4'] = pivot['rel_1e3'] - pivot['rel_1e4']
        print(metric); display(pivot); print('Paired mean delta:', pivot['delta_1e3_minus_1e4'].mean())

if selected_reliability is None and len(reliability_common) >= 2:
    selected_reliability = 0.001 if np.mean([candidate_scores[s] - baseline_scores[s] for s in reliability_common]) > 0 else 0.0001
if selected_reliability is not None:
    anchor_prefix = RELIABILITY_PREFIX if selected_reliability == 0.001 else BASELINE_PREFIX
    weight_conditions = {'w0p10': anchor_prefix, 'w0p05': WEIGHT_RUNS[selected_reliability][0.05][0], 'w0p20': WEIGHT_RUNS[selected_reliability][0.20][0]}
    weight_loaded = {name: load_condition(name, prefix) for name, prefix in weight_conditions.items()}
    validation = pd.DataFrame([row for value in weight_loaded.values() for row in value[0]])
    seed_sets = [set(validation.query('condition == @name')['seed']) for name in weight_conditions] if not validation.empty else []
    common = sorted(set.intersection(*seed_sets)) if seed_sets else []
    print('SupCon-weight common paired seeds:', common, 'fixed reliability:', selected_reliability)
    if len(common) < 2: print('EXPLORATORY/INCOMPLETE: two common paired seeds are required for the initial decision.')
    paired = validation.query('seed in @common').copy(); display(paired)
    metrics = ['validation_macro_f1', 'validation_accuracy', 'validation_weighted_f1', 'validation_macro_precision', 'validation_macro_recall', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']
    for metric in metrics:
        pivot = paired.pivot(index='seed', columns='condition', values=metric)
        for condition in ('w0p05', 'w0p20'): pivot[f'{condition}_minus_w0p10'] = pivot[condition] - pivot['w0p10']
        print(metric); display(pivot); print('Condition means:', paired.groupby('condition')[metric].mean().to_dict())
    all_test = [frame for value in weight_loaded.values() for frame in value[1]]
    if common and all_test:
        test = pd.concat(all_test, ignore_index=True).query('seed in @common')
        test_metrics = [column for column in ['accuracy', 'micro_f1', 'weighted_f1', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if column in test]
        print('=== Locked-test descriptive metrics; never used for selection ==='); display(test.groupby('condition')[test_metrics].agg(['mean', 'std']))
        print('=== Per-dataset test macro-F1 ==='); display(pd.concat([frame for value in weight_loaded.values() for frame in value[3]], ignore_index=True).query('seed in @common').pivot_table(index='dataset', columns='condition', values='macro_f1', aggfunc='mean'))
        print('=== Per-class test F1 ==='); display(pd.concat([frame for value in weight_loaded.values() for frame in value[2]], ignore_index=True).query('seed in @common').pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    resources = [frame for value in weight_loaded.values() for frame in value[4]]
    if resources:
        resources = pd.concat(resources, ignore_index=True); display(resources)
        assert set(resources['total_parameters'].astype(int)) == {22420} and set(resources['trainable_parameters'].astype(int)) == {2300}
    latent = [frame for value in weight_loaded.values() for frame in value[5]]
    if latent:
        latent = pd.concat(latent, ignore_index=True).query('seed in @common')
        print('=== Stage-A latent linear-probe and kNN metrics ==='); display(latent.groupby(['condition', 'probe', 'target'])[['accuracy', 'balanced_accuracy', 'macro_f1_present']].mean())


## Routing, expert-preservation, and loss-share diagnostics

In [ ]:
diagnostic_prefixes = {'rel_1e4': BASELINE_PREFIX, 'rel_1e3': RELIABILITY_PREFIX}
if selected_reliability is not None:
    diagnostic_prefixes.update({'w0p05': WEIGHT_RUNS[selected_reliability][0.05][0], 'w0p20': WEIGHT_RUNS[selected_reliability][0.20][0]})
for condition, prefix in diagnostic_prefixes.items():
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        if not (result_dir / 'manifest.json').is_file(): continue
        checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{prefix}_seed{seed}'
        stage_b_path = checkpoint_dir / 'Validation_Expert_Owned_StageB.csv'; stage_c_path = result_dir / 'Validation_Expert_Owned_StageC.csv'
        if stage_b_path.is_file() and stage_c_path.is_file():
            preserved = pd.read_csv(stage_b_path).merge(pd.read_csv(stage_c_path), on='dataset', suffixes=('_stage_b', '_stage_c')); delta_columns = []
            for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
                column = f'delta_{metric}'; preserved[column] = preserved[f'{metric}_stage_c'] - preserved[f'{metric}_stage_b']; delta_columns.append(column)
            assert np.allclose(preserved[delta_columns], 0.0, atol=1e-12), f'Frozen body drift in {condition}, seed {seed}'
            print(f'=== {condition}, seed {seed}: exact Stage-B expert preservation ==='); display(preserved)
        print(f'=== {condition}, seed {seed}: gate entropy/confidence ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
        print(f'=== {condition}, seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))
        print(f'=== {condition}, seed {seed}: class reliability ==='); display(pd.read_csv(result_dir / 'Class_Conditional_Reliability.csv'))
        history_path = result_dir / 'training' / 'Training_History.csv'
        if history_path.is_file():
            history = pd.read_csv(history_path)
            columns = [column for column in ['stage', 'epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_representation_loss', 'train_balance_penalty', 'train_reliability_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if column in history]
            display(history[columns])


## Decision rule

Reliability is selected first by paired mean validation macro-F1; a tie retains `1e-4`. At the selected reliability, choose among SupCon weights `0.05`, `0.10`, and `0.20` using paired mean validation macro-F1. Treat a candidate as a practical improvement only when the mean gain is about `0.003` or larger, its direction is reasonably consistent across seeds, validation accuracy declines by no more than `0.001`, worst-dataset macro-F1 declines by no more than `0.01`, and rare-class recall remains close to the Notebook-42 level. Confirm only the selected configuration with seed 2; each runner produces `Final_Study_Manifest.json` after its full three-seed confirmation. If neither bracket beats `0.10`, retain Notebook 42 and stop this line of tuning. Locked-test results support interpretation but never choose the setting.